In [ ]:
from datetime import datetime
import pickle 
import ipinfo
import openmeteo_requests
import requests_cache
from retry_requests import retry
import unicodedata


current_date = datetime.today().strftime('%Y-%m-%d')
with open("current_date.txt", "r") as f:
    old_date = f.read()

if(old_date != current_date):
    print("New Day!, Updating Details...")
    
    access_token = 'f99daaa4badd1b'
    handler = ipinfo.getHandler(access_token)
    details = handler.getDetails()
    with open('ip_details.pkl', 'wb') as f:
        pickle.dump(details.details, f)
            

with open('ip_details.pkl', 'rb') as f:
        ip_details = pickle.load(f)

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": ip_details["latitude"],
	"longitude": ip_details["longitude"],
	"current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature", "is_day", "precipitation", "rain", "showers"],
	"timezone": "auto",
	"forecast_days": 1
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]


# Current values. The order of variables needs to be the same as requested.
current = response.Current()

current_temperature = int(current.Variables(0).Value())

current_relative_humidity_2m = current.Variables(1).Value()

current_apparent_temperature = current.Variables(2).Value()

current_is_day = current.Variables(3).Value()

current_precipitation = current.Variables(4).Value()

current_rain = current.Variables(5).Value()

current_showers = current.Variables(6).Value()

# print(f"Current time {current.Time()}")
city = ip_details["city"]
city  = unicodedata.normalize('NFD', city).encode('ascii', 'ignore').decode('ASCII')
region = ip_details["region"]
print(f"Current Temperature in {city}, {region} is {current_temperature}°C")
# print(f"Current relative_humidity_2m {current_relative_humidity_2m}")
# print(f"Current apparent_temperature {current_apparent_temperature}")
# print(f"Current is_day {current_is_day}")
# print(f"Current precipitation {current_precipitation}")
# print(f"Current rain {current_rain}")
# print(f"Current showers {current_showers}")


New Day!, Updating Details...
Current Temperature in Phagwara, Punjab is 19°C
